# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`). Each `@id` allows referencing specific entities in the Croissant schema.

In [ ]:
# List record sets and their fields by @id
print('Record sets:')
for record_set in metadata.record_sets:
    print(f"- Record Set @id: {record_set.id}, name: {record_set.name}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - Field @id: {field.id}, name: {field.name}, datatype: {field.data_type}")
    print()

## 2.1 Inspecting First Few Records per RecordSet
Each record set contains tabular data with well-defined fields. We can load and preview the first few records. Use the `@id` of the desired record set.

In [ ]:
# Inspect example records from EACH record set
for record_set in metadata.record_sets:
    print(f'Record set: {record_set.name} (@id={record_set.id})')
    # show first example records
    for i, rec in enumerate(dataset.records(record_set=record_set.id)):
        print(rec)
        if i>=2:
            break
    print('-'*80)

## 3. Data Extraction
Load data from available record sets into Pandas DataFrames for further analysis. Each record set is referenced by its `@id`. Often, the main experimental data is in one primary record set, but auxiliary sets may contain reference tables.

All following steps will rely on the dynamic discovery of available record set IDs.

In [ ]:
# Extract all record sets into DataFrames, keys are RecordSet @id

record_sets_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for rs_id in record_sets_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for record set {rs_id}")

if record_sets_ids:
    print(f'Columns in first record set ({record_sets_ids[0]}):')
    print(dataframes[record_sets_ids[0]].columns.tolist())
    display(dataframes[record_sets_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping on the main experimental data. Choose numeric and group-by fields by their `@id`, as listed above.

In [ ]:
# Choose the main record set and numeric/grouped fields

# If multiple record sets, select the main record set (usually the first one)
main_record_set_id = record_sets_ids[0] if record_sets_ids else None
df = dataframes[main_record_set_id]

# Inspect available columns (by @id, which are keys in the record dicts)
print(f'Available columns (@id) in {main_record_set_id}:')
print(df.columns.tolist())

# Try to infer a numeric field (many proteomics datasets have MW, coverage, abundance, or peptides fields)
possible_numeric = [col for col in df.columns if df[col].dtype in ['int64','float64']]
if not possible_numeric:
    # Try parsing columns with numeric content
    for c in df.columns:
        try:
            df[c+'_float'] = pd.to_numeric(df[c], errors='coerce')
            if df[c+'_float'].notna().sum()>0:
                possible_numeric.append(c+'_float')
        except Exception:
            continue

if possible_numeric:
    numeric_field = possible_numeric[0]
else:
    numeric_field = df.columns[0]

print(f'Chosen numeric field: {numeric_field}')

# Try to find a grouped field (e.g., a sample, condition, or protein category)
group_field = None
for col in df.columns:
    if 'group' in col.lower() or 'type' in col.lower() or 'category' in col.lower():
        group_field = col
        break

threshold = df[numeric_field].quantile(0.75) if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
# Filter
filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce') > threshold]
print(f"Filtered records from {main_record_set_id} with {numeric_field} > {threshold} (n={filtered_df.shape[0]}):")
display(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group
if group_field and group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of a numeric field or its relationship to a group/category, if available.

In [ ]:
# Visualize the numeric field, if available
import matplotlib.pyplot as plt
import seaborn as sns

if possible_numeric:
    plt.figure(figsize=(7,4))
    sns.histplot(pd.to_numeric(df[numeric_field], errors='coerce').dropna(), bins=40)
    plt.title(f"Distribution of {numeric_field} in record set {main_record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    if group_field is not None and group_field in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
In this notebook, we loaded a Croissant-described dataset using `mlcroissant`, examined its available record sets, and performed exploratory analysis on a numeric field (chosen by inspecting field `@id`s). This workflow can be extended to perform more advanced analyses, link to external datasets, or integrate with downstream bioinformatics pipelines.